# 04 Statistical Analysis


In [ ]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    markers = ("src", "data", "notebooks")

    for p in [start, *start.parents]:
        if p.name == "customer-segmentation-analytics" and all((p / m).is_dir() for m in markers):
            return p

    for p in [start, *start.parents]:
        candidate = p / "Improvements" / "Statistics" / "customer-segmentation-analytics"
        if all((candidate / m).is_dir() for m in markers):
            return candidate

    raise RuntimeError("Could not locate customer-segmentation-analytics project root.")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"PROJECT_ROOT: {PROJECT_ROOT}")


## Objective


Test whether behaviour and value metrics differ significantly across customer groups.


In [ ]:
import pandas as pd
from scipy import stats

features = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "customer_features.parquet")
features.head()


## Test 1: Spearman Correlation (Purchase Frequency vs Spend)


In [ ]:
rho, p_corr = stats.spearmanr(features["purchase_frequency"], features["avg_monthly_spend"])
{"spearman_rho": float(rho), "p_value": float(p_corr)}


## Test 2: ANOVA (Spend Across Provided Segments)


In [ ]:
groups = [
    group["avg_monthly_spend"].values
    for _, group in features.groupby("customer_segment")
    if len(group) > 1
]

anova_stat, anova_p = stats.f_oneway(*groups)
{"anova_stat": float(anova_stat), "p_value": float(anova_p), "n_groups": len(groups)}


## Test 3: Chi-Square (Payment Method vs Segment)


In [ ]:
contingency = pd.crosstab(features["payment_method"], features["customer_segment"])
chi2, p_chi, dof, expected = stats.chi2_contingency(contingency)
{"chi2": float(chi2), "p_value": float(p_chi), "dof": int(dof)}


## Test 4: High vs Low Discount Users (Order Value Difference)


In [ ]:
threshold = features["discount_usage_rate"].median()
high_disc = features.loc[features["discount_usage_rate"] > threshold, "avg_order_value"]
low_disc = features.loc[features["discount_usage_rate"] <= threshold, "avg_order_value"]

t_stat, t_p = stats.ttest_ind(high_disc, low_disc, equal_var=False, nan_policy="omit")
{"t_stat": float(t_stat), "p_value": float(t_p)}


## Interpretation


Summarize significance and commercial impact in plain language for each test.
